In [1]:
import torch
import torch.nn as nn
import time

# Configuração da mesma MLP do código CUDA
input_size = 784
hidden_1 = 128
hidden_2 = 64
output_size = 10
batches = [1, 64, 256, 1024]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Rodando PyTorch em: {device}")

# Modelo equivalente
model = nn.Sequential(
    nn.Linear(input_size, hidden_1),
    nn.ReLU(),
    nn.Linear(hidden_1, hidden_2),
    nn.ReLU(),
    nn.Linear(hidden_2, output_size)
).to(device).eval()

# Loop de Benchmarks
for batch in batches:
    # Cria dado aleatório simulando o lote
    x = torch.randn(batch, input_size, device=device)

    # Warm-up (essencial em GPU para estabilizar clocks)
    for _ in range(10):
        _ = model(x)

    torch.cuda.synchronize()

    # Medição de tempo precisa
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(100): # Roda 100 vezes para tirar a média
            out = model(x)
    torch.cuda.synchronize()
    end = time.perf_counter()

    ms_medio = ((end - start) * 1000) / 100
    print(f"Batch Size {batch:4d} | Tempo Médio PyTorch: {ms_medio:.4f} ms")

Rodando PyTorch em: cpu


AssertionError: Torch not compiled with CUDA enabled